# 26 — Evaluation Baselines (Smoke Test)

Compares baselines on a **~10-question subset** of the eval set (keeps API cost manageable on Groq).

| Baseline | Meaning |
|----------|--------|
| `vanilla_rag` | single retrieve → answer |
| `facet_icot` | iterative facet-aware ICOT-RAG |

Hard metrics:
- **facet recall** — required evidence facets present in retrieved docs
- **keyword hit rate** — gold `reference_hints` found in retrieved text
- **source hit rate** — expected KB sources present
- source diversity, iterations, doc count

LLM-as-judge scores come later.

In [ ]:
import os
import sys
import json

project_root = os.path.abspath("..")

if project_root not in sys.path:
    sys.path.insert(0, project_root)

In [ ]:
from rag_icot.evaluation import (
    load_eval_dataset,
    run_vanilla_rag,
    run_facet_icot,
    summarize_run,
)
from rag_icot.components.retriever import Retriever
from rag_icot.components.answer_generator import AnswerGenerator
from rag_icot.pipeline.rag_icot_pipeline import RAGICOTPipeline

In [ ]:
dataset_path = os.path.join(
    project_root,
    "datasets",
    "evaluation",
    "iot_security_eval_v1.json",
)

questions = load_eval_dataset(dataset_path)

# ~10 mixed categories: behaviour / vuln / exploit / technique /
# mitigation / multi-facet / faithfulness
smoke_ids = [
    "q001",  # Mirai behaviour
    "q002",  # Torii behaviour
    "q011",  # CVE-2020-8863
    "q012",  # CVE-2018-12153
    "q019",  # exploit / DNS routers
    "q021",  # MITRE technique
    "q025",  # MITRE mitigation
    "q031",  # Mirai + MITRE multi-facet
    "q032",  # Mirai + mitigation multi-facet
    "q041",  # faithfulness (KB likely insufficient)
]
by_id = {q.id: q for q in questions}
smoke = [by_id[i] for i in smoke_ids if i in by_id]

print(f"Smoke size: {len(smoke)} questions")
for q in smoke:
    print(
        q.id,
        "|",
        q.category,
        "|",
        q.expected_sources,
        "|",
        q.question,
    )

In [ ]:
# Reuse heavy clients across questions
retriever = Retriever()
generator = AnswerGenerator()
pipeline = RAGICOTPipeline()
print("LLM ready:", generator.llm.provider, generator.llm.model_name)

In [ ]:
import time

rows = []
errors = []

# Short pause helps Groq free-tier rate limits between questions
PAUSE_SECONDS = 8

for q in smoke:
    print("\n" + "=" * 80)
    print("QUESTION", q.id, q.question)
    print("=" * 80)

    try:
        vanilla = run_vanilla_rag(
            q.question,
            k=5,
            retriever=retriever,
            generator=generator,
        )
        icot = run_facet_icot(
            q.question,
            max_iterations=2,
            pipeline=pipeline,
        )

        v_sum = summarize_run(
            vanilla,
            required_facets=q.required_facets,
            expected_sources=q.expected_sources,
            reference_hints=q.reference_hints,
        )
        i_sum = summarize_run(
            icot,
            required_facets=q.required_facets,
            expected_sources=q.expected_sources,
            reference_hints=q.reference_hints,
        )

        print("vanilla:", v_sum)
        print("facet_icot:", i_sum)

        rows.append({
            "id": q.id,
            "category": q.category,
            "required_facets": q.required_facets,
            "expected_sources": q.expected_sources,
            "reference_hints": q.reference_hints,
            "vanilla": v_sum,
            "facet_icot": i_sum,
            "vanilla_answer_preview": vanilla["answer"][:400],
            "icot_answer_preview": icot["answer"][:400],
        })

    except Exception as exc:
        print(f"FAILED {q.id}: {type(exc).__name__}: {exc}")
        errors.append({
            "id": q.id,
            "error": f"{type(exc).__name__}: {exc}",
        })

    print(f"Sleeping {PAUSE_SECONDS}s before next question...")
    time.sleep(PAUSE_SECONDS)

print(f"\nCompleted: {len(rows)} | Failed: {len(errors)}")
if errors:
    print(errors)

In [ ]:
# Comparison table: facet recall + keyword + source hits
if not rows:
    print("No successful rows yet — re-run the evaluation cell above.")
else:
    header = (
        f"{'ID':<6} {'Cat':<14} "
        f"{'V_fac':>6} {'I_fac':>6} "
        f"{'V_kw':>6} {'I_kw':>6} "
        f"{'V_src':>6} {'I_src':>6}"
    )
    print(header)
    print("-" * len(header))

    for row in rows:
        v = row["vanilla"]
        i = row["facet_icot"]
        print(
            f"{row['id']:<6} {row['category']:<14} "
            f"{v['facet_recall']:>6.2f} {i['facet_recall']:>6.2f} "
            f"{v['keyword_hit_rate']:>6.2f} {i['keyword_hit_rate']:>6.2f} "
            f"{v['source_hit_rate']:>6.2f} {i['source_hit_rate']:>6.2f}"
        )

    def avg(key, method):
        return sum(r[method][key] for r in rows) / len(rows)

    print("-" * len(header))
    print(
        f"{'AVG':<6} {'':<14} "
        f"{avg('facet_recall','vanilla'):>6.2f} {avg('facet_recall','facet_icot'):>6.2f} "
        f"{avg('keyword_hit_rate','vanilla'):>6.2f} {avg('keyword_hit_rate','facet_icot'):>6.2f} "
        f"{avg('source_hit_rate','vanilla'):>6.2f} {avg('source_hit_rate','facet_icot'):>6.2f}"
    )

In [ ]:
# Save smoke results for later analysis / paper tables
out_path = os.path.join(
    project_root,
    "artifacts",
    "evaluation",
    "smoke_baseline_results.json",
)

os.makedirs(os.path.dirname(out_path), exist_ok=True)

payload = {
    "rows": rows,
    "errors": errors if "errors" in dir() else [],
}

with open(out_path, "w", encoding="utf-8") as f:
    json.dump(payload, f, indent=2, ensure_ascii=False)

print("Saved", out_path)
print("Successful rows:", len(payload["rows"]))
print("Errors:", len(payload["errors"]))

## Next

1. LLM-as-judge scoring (reliability / relevance / technicality)
2. Ablations: `max_iter=1` vs `3`, no source filter / no facets
3. Full 50-question batch once smoke metrics look stable
4. Optional: Zeng-style prompt-only ICoT baseline